## Scientific Validation

This notebook acts as a **scientific validation and documentation layer** on top of the already existing Machine Learning pipeline of the **Math-Music-Lab** project.

It does not replace the main Python files. Instead, it complements them:

- `math_music_data_cleaning.py` — data cleaning
- `math_music_eda_feature_engineering.py` — exploratory data analysis and feature engineering
- `math_music_modeling_and_mlflow.py` — modeling and MLflow tracking

The notebook focuses on aspects such as:

- whether the results are reproducible;
- where the data comes from and why it is suitable for this project;
- whether the target variable is properly defined;
- whether there is any risk of data leakage;
- comparison against a baseline model;
- model interpretation;
- error analysis;
- limitations of the project and possible improvements.

### Important Reproducibility Note

This notebook should be executed **at least two times**.

On the first run, it creates the following file:

`data/processed/data_version_manifest.json`

This file serves as the initial version record of the processed datasets.

On the second and later runs, the notebook compares the current processed datasets against the previously saved manifest. This makes it possible to check whether the data has remained unchanged or has been modified.

However, for convenience, the project will also include the necessary data required specifically to run this notebook. These files will be placed in:

`Math-Music-Lab/data/processed/` and `Math-Music-Lab/data/master_dataset_final.csv`

This allows the notebook to be executed directly, without requiring the user to manually prepare or generate all input files beforehand.

## Project Role and Scientific Framing

Main research question:

**Can the mathematical fingerprint of a song, represented through its audio features, help predict whether the song becomes a mainstream Billboard hit?**

In this project, each song is represented as a numerical vector of audio features:

$$
x = \text{audio features of a song}
$$

The target variable is a binary indicator:

$$
y =
\begin{cases}
1, & \text{if the song is a Billboard hit} \\
0, & \text{otherwise}
\end{cases}
$$

The goal of the model is to estimate whether the audio-feature vector contains useful information for predicting the probability that a song becomes a hit:

$$
f(x) \approx P(y = 1 \mid x)
$$

In simpler terms, the model tries to answer the question:

**Given only the measurable audio characteristics of a song, how likely is it that the song belongs to the Billboard-hit class?**

### Hypotheses

The **null hypothesis** is that the audio features do not contain enough useful predictive information.

If the null hypothesis is true, then we expect the model not to perform better than a very simple comparison model. For example, such a model could:

- choose the prediction randomly;
- always predict the most common result in the dataset, for example “not a hit”;
- be implemented with `DummyClassifier` from `scikit-learn`, which is commonly used as a simple baseline for comparison.

The **alternative hypothesis** is that the audio features do contain some useful information.

In other words, features such as tempo, energy, danceability, loudness, and others may help the model predict Billboard hits slightly better than such a simple baseline model.

### Scope and Limitations

This analysis is intentionally limited to audio-based information.

Billboard success depends on many factors that are not included in the dataset, such as:

- artist popularity;
- marketing and promotion;
- cultural timing;
- playlist placement;
- social media trends;
- genre popularity;
- radio exposure;
- collaborations and label support.

Because of this, the model should not be interpreted as a complete explanation of why a song becomes successful.

Instead, the goal is more modest:

**to test whether audio features alone contain a weak but measurable signal related to Billboard-hit prediction.**

In [ ]:
print("=" * 80)
print("Scientific Project Check")
print("=" * 80)
print("This file extends the existing pipeline with reproducibility, validation,")
print("interpretability, and error analysis.")
print("=" * 80)

## 1. Imports, Configuration, and Reproducibility Checks

This section verifies that the expected processed datasets exist and records key reproducibility information: paths, file checksums, dataset shapes, target distribution, package versions, and random seed.

It also creates or compares the data-version manifest file:

`data/processed/data_version_manifest.json`

On the first run, the manifest is created. On later runs, the current datasets are compared against the saved baseline.

In [ ]:
import sys
import platform
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path(".").resolve()

MASTER_DATASET_PROCESSED = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
MASTER_DATASET_FINAL = PROJECT_ROOT / "data" / "master_dataset_final.csv"
CORGIS_HISTORICAL_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corgis_historical.csv"

REQUIRED_FILES = {
    "master_dataset_processed": MASTER_DATASET_PROCESSED,
    "master_dataset_final": MASTER_DATASET_FINAL,
    "corgis_historical_processed": CORGIS_HISTORICAL_PROCESSED,
}


def compute_file_checksum(path, algorithm="sha256"):
    """
    Compute a checksum for a file.

    The checksum is used as a simple data-versioning mechanism.
    If the source URLs change or the data is regenerated differently,
    the checksum will change as well.
    """
    hash_object = hashlib.new(algorithm)

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            hash_object.update(chunk)

    return hash_object.hexdigest()


print("\n" + "=" * 80)
print("1. Reproducibility checks")
print("=" * 80)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Random seed: {RANDOM_SEED}")

print(
    "\nEnvironment note: these versions are recorded for reproducibility. "
    "For a submitted repository, the same dependencies should also be listed "
    "in requirements.txt."
)

print("\nChecking required result files:")

missing_files = []

for name, path in REQUIRED_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        checksum = compute_file_checksum(path)
        print(f"  OK: {name}")
        print(f"      Path: {path}")
        print(f"      Size: {size_mb:.2f} MB")
        print(f"      SHA256: {checksum[:16]}...")
    else:
        print(f"  MISSING: {name}")
        print(f"      Expected path: {path}")
        missing_files.append(path)

if missing_files:
    raise FileNotFoundError(
        "Some required files are missing. "
        "Please run the original three scripts before running this validation file."
    )

print("\nAll required result files are available.")

df_master = pd.read_csv(MASTER_DATASET_PROCESSED)
df_final = pd.read_csv(MASTER_DATASET_FINAL)
df_corgis = pd.read_csv(CORGIS_HISTORICAL_PROCESSED)

print("\nLoaded datasets:")
print(f"  df_master: {df_master.shape}")
print(f"  df_final:  {df_final.shape}")
print(f"  df_corgis: {df_corgis.shape}")

if "is_hit" not in df_master.columns:
    raise ValueError("Column 'is_hit' is missing from df_master.")

if "is_hit" not in df_final.columns:
    raise ValueError("Column 'is_hit' is missing from df_final.")

print("\nTarget distribution in df_master:")
print(df_master["is_hit"].value_counts(dropna=False))
print(f"Hit rate: {df_master['is_hit'].mean() * 100:.2f}%")

print("\nTarget distribution in df_final:")
print(df_final["is_hit"].value_counts(dropna=False))
print(f"Hit rate: {df_final['is_hit'].mean() * 100:.2f}%")

BASE_AUDIO_FEATURES = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

missing_audio_features = [
    col for col in BASE_AUDIO_FEATURES if col not in df_master.columns
]

if missing_audio_features:
    raise ValueError(
        f"The following expected audio features are missing: {missing_audio_features}"
    )

print("\nBase audio features available:")
for col in BASE_AUDIO_FEATURES:
    print(f"  - {col}")

print("\nReproducibility check complete.")